# 04 — Structured Outputs and Typed Interfaces

## Scenario
Northstar must convert a support request into a structured `CaseBrief` for a review queue. The application needs a parseable interface with explicit fields for intent, summary, evidence, and recommendation.

**Safety boundary:** Provider-native structured output can constrain shape, but the application must still handle parse failures, refusals, truncation, unsupported schema features, and semantic errors. The replay path demonstrates valid JSON with fabricated evidence and a bounded repair loop that applies the same deterministic checks used by the optional live path.

In [ ]:
from pathlib import Path
from northstar.contracts import parse_structured
from northstar.runtime import get_client
from lab04 import CaseBrief, build_requests, bounded_repair, generate_case_brief, validate_evidence

client = get_client(Path("fixtures/replays.json"))
REQUESTS = {request.case_id: request for request in build_requests()}

def show(case_id):
    request = REQUESTS[case_id]
    print(f"\n=== {case_id} ===")
    print("SYSTEM:\n" + (request.system or "(none)"))
    for message in request.messages:
        print(f"{message.role.upper()}:\n{message.text}")
    response = client.generate(request)
    print("RECORDED RESPONSE:\n" + response.text)
    result = parse_structured(CaseBrief, response.text)
    parsed = result.value if result.ok else None
    print("PARSED VALUE:", parsed.model_dump() if parsed else None)
    print("PARSE ERROR:", result.error)
    return request, response, parsed, result

## Demonstration 1: Syntax Guarantees

Schema-constrained generation improves shape adherence. This recorded case is parseable and conforms to `CaseBrief`; the later malformed case proves that the application must still keep an explicit parse-failure path.

In [ ]:
_, _, brief, result = show("b04/syntax/good")
assert result.ok
assert isinstance(brief, CaseBrief)

## Demonstration 2: Semantic Failure (Hallucination)

However, structural correctness does NOT mean factual correctness. Let's see what happens if the user tricks the model into citing a fake policy.

In [ ]:
_, _, hallucinated, result = show("b04/semantic/hallucinated")
assert result.ok
assert validate_evidence(hallucinated) == "unknown_evidence_id"

## Demonstration 3: Application-Side Validation and Bounded Repair

Because we cannot trust the model's semantics, we must validate business rules in standard code (e.g., Python). If validation fails, we can attempt a bounded repair.

In [ ]:
repaired, attempts, terminal = bounded_repair(client)
print("Repair attempts:", attempts, "terminal:", terminal)
assert attempts == 2
assert repaired is not None
assert repaired.evidence_cited == "pol_return_30d"
assert validate_evidence(repaired) is None

## Takeaway

The assertions above describe the deterministic recorded run. Change one prompt variable, rerun the lab, and measure the trade-off.

## References

See the course README for the folded reference material and links.

## Reading the typed-interface experiment

The first demonstration separates syntax from meaning. The recorded response is
valid JSON that Pydantic can parse into `CaseBrief`, but that fact alone does not
prove that the recommended action is supported. Typed outputs make downstream
handling predictable; they do not grant the model authority to invent a policy.

The tricky request names `pol_elite_instant_refund`, a plausible-looking policy
that is not in the approved set. This is a semantic failure even though all
required fields and literal values have the right shape. The application validator
returns `unknown_evidence_id`, and the notebook treats that result as the
important observation rather than trusting the fluent explanation.

Repair is bounded and evidence-driven. Attempt one receives the invalid evidence
ID; the next request includes a narrow error message and returns the approved
`pol_return_30d` evidence for its refund intent. The exhausted path demonstrates
the other safe outcome: after two
invalid attempts, the system returns no brief and routes to `human_review`.
Unbounded self-repair would turn a malformed response into a cost and latency
incident. The malformed replay is a separate failure class: parsing fails with
`not_json`, and the application reports the error instead of raising through the
lesson.

The repair cells deliberately show both the invalid intermediate object and the
terminal decision. In production, retain these attempts in an audit record with
the validator error, prompt fingerprint, and escalation reason. A successful
repair is evidence that the bounded workflow worked; it is not evidence that the
original model response was safe to use. The approved evidence set is deliberately
small so that a learner can inspect every allowed identifier. Expanding the schema
or adding a provider-native response mode should be a later experiment, after the
application-side validator and terminal states are covered by tests.
That ordering keeps failure handling observable instead of implicit.